# Task 1: Fashion Item Type Classification

**COSC2753 Machine Learning, Assignment 2 (2026B)** &nbsp;|&nbsp; **Google Colab edition**

## Executive Summary

- **Problem.** Predict `articleType` from a 60x80 catalogue image. 37,846 labelled rows and 124 observable classes, with support running from 6,780 images (Tshirts) down to 1. The long tail, not the label space size, is what makes this task hard.
- **Evaluation first.** Section 2 fixes the split, the metrics, and the baselines before any model is fitted. Every later number is scored by the same function on the same rows, so the comparison table is a like-for-like one.
- **Primary metric: macro-F1** over the classes present in validation, reported beside top-1 accuracy, balanced accuracy, top-5 accuracy, and macro-F1 broken down by class support bucket. A single aggregate number cannot separate "learns the head" from "learns the tail" on this distribution.
- **Three models, one rung each.** HOG + linear SVM (does this need deep learning?), a plain CNN trained from scratch (the reference point), and a small ResNet with decoupled classifier retraining (the advanced model). The CNN is kept deliberately plain so the gain of the third model is attributable.
- **Ablations, not extra models.** The imbalance treatments (plain cross-entropy, logit-adjusted cross-entropy, class-balanced classifier retraining) are three rows measured on one architecture, not three separate submissions.
- **Ultimate judgement.** Section 8 goes past the metric table: confusion structure checked against the visual-confusability hypotheses from Section 2.3 of the preprocessing notebook, per-class F1 against support, hierarchical error severity, calibration, and inference cost.

This is the Colab edition. It is identical to the local notebook from Section 2 onward; only Section 1 differs, where it mounts Drive, extracts the dataset archive, and discovers the paths that the local edition takes from the repository layout.

All preprocessing decisions are inherited from `00_eda_and_preprocessing.ipynb`. This notebook makes no data decision of its own; it reads the audited manifest and follows the five handover rules in Section 5 of that notebook.

## How to Run on Colab

### Before you start

1. **Runtime > Change runtime type > T4 GPU.** The notebook runs on CPU but takes several hours
   instead of under an hour.
2. Open the file browser in the left sidebar (the folder icon) and drag three files into
   `/content`, the top level:
   - `A2_Fashion.zip`, the archive from Canvas
   - `preprocessing.py`, the shared data access module
   - `train_manifest.csv`, written by Section 3.4 of `00_eda_and_preprocessing.ipynb`

   The manifest is not in the archive. This notebook consumes the audit, it does not perform
   one, so run notebook 00 first and bring its output across.
3. Wait for the uploads to finish before running anything. The zip is the slow one, and a
   partially uploaded archive fails at extraction with a confusing error rather than a clear one.

Then run the notebook top to bottom. Section 1.2 extracts the archive and Section 1.3 discovers
every path by searching the extracted tree, so the internal layout of the zip does not matter.

If you prefer a dialog to drag and drop, set `USE_UPLOAD_WIDGET = True` in Section 1.2.

### Runtime controls

| Flag | Effect |
|---|---|
| `QUICK_RUN` | 3 epochs, one seed, 5,000 training rows. Meaningless numbers, but it exercises every cell in a few minutes. Run this first. |
| `RUN_SEED_STUDY` | Set `False` to skip the multi-seed variance runs, the single most expensive section. |
| `RUN_SALIENCY` | Set `False` to skip the gradient saliency figures. |

### Two Colab-specific warnings

- **The session disk is temporary.** Everything under `/content`, including the three files you
  uploaded, disappears when the runtime disconnects. Section 9.1 downloads the trained model and
  the predictions to your machine; do not skip it, or a 50-minute run is lost. The uploads have
  to be repeated in the next session.
- **Free-tier sessions idle out.** Roughly 90 minutes of inactivity, and a hard cap of around 12
  hours. The full notebook fits inside that window on a T4, but do not leave it unattended near
  the end of a session.

### What it produces

- `models/task1/` : the selected model's weights, the class-index mapping, the normalisation
  constants, and the run configuration.
- `outputs/task1_predictions.csv` : an `articleType` prediction for every row of
  `styles_prediction.csv`, to be merged with the other tasks' columns before submission.

Section 9.1 downloads both.

## 1. Setup

### 1.1 Runtime and Dependencies

Colab ships with PyTorch, scikit-learn and scikit-image already installed, so this cell normally
reports that there is nothing to do. It is kept because Colab's preinstalled set changes without
notice, and a missing package should fail here with a readable message rather than thirty cells
later inside a training loop.

In [ ]:
import importlib
import subprocess
import sys

REQUIRED = {
    "torch": "torch",
    "torchvision": "torchvision",
    "sklearn": "scikit-learn",
    "skimage": "scikit-image",
    "pandas": "pandas",
    "seaborn": "seaborn",
}

missing = [package for module, package in REQUIRED.items()
           if importlib.util.find_spec(module) is None]
if missing:
    print("Installing:", ", ".join(missing))
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", *missing], check=True)
else:
    print("All dependencies already present.")

import torch

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0),
          f"| {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
else:
    print("No GPU. Runtime > Change runtime type > T4 GPU, or expect several hours; "
          "set QUICK_RUN = True in Section 1.4 first.")

In [ ]:
%matplotlib inline

import gc
import json
import math
import sys
import time
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from IPython.display import display

import torch
import torch.nn as nn
import torch.nn.functional as F

from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    confusion_matrix,
    f1_score,
)
from sklearn.svm import LinearSVC

warnings.filterwarnings("ignore", category=FutureWarning)
warnings.filterwarnings("ignore", category=UserWarning)

pd.set_option("display.max_columns", 40)
pd.set_option("display.width", 160)
sns.set_theme(style="whitegrid", context="notebook")

PALETTE = ["#2a78d6", "#eb6834", "#1baf7a", "#eda100", "#8b6fc0", "#e87ba4"]
MUTED = "#6b7280"

### 1.2 Extracting the Uploaded Archive

Three files are expected at `/content`, dragged into the file browser in the left sidebar:
`A2_Fashion.zip`, `preprocessing.py` and `train_manifest.csv`. This cell checks for them and
extracts the archive onto the session disk.

The archive is extracted rather than read in place because `zipfile` cannot serve 38,000 random
reads efficiently, and the deep models touch every image once per epoch. Extraction costs a minute
and is paid once; reading from the archive would be paid on every epoch.

Re-running is safe. Extraction is skipped only when the target directory already holds files, so
a directory left behind by an interrupted or failed unzip is removed and redone rather than
mistaken for a finished job.

Set `USE_UPLOAD_WIDGET = True` to be prompted with a file dialog instead of using the sidebar.
Both routes put the files in the same place; the widget is slower for a file the size of the
archive and is offered only as a fallback.

In [ ]:
import shutil
import zipfile

# ---------------------------------------------------------------------------------
USE_UPLOAD_WIDGET = False        # True: prompt with a file dialog instead of the sidebar
ARCHIVE_NAME = "datasets.zip"       # match the name in the file browser exactly
# ---------------------------------------------------------------------------------

WORK_ROOT = Path("/content")
EXTRACT_ROOT = WORK_ROOT / "dataset_extracted"
REQUIRED_UPLOADS = [ARCHIVE_NAME, "preprocessing.py"]   # the manifest is checked in 1.3

if USE_UPLOAD_WIDGET:
    from google.colab import files
    print("Select", ", ".join(REQUIRED_UPLOADS + ["train_manifest.csv"]))
    files.upload()

absent = [name for name in REQUIRED_UPLOADS if not (WORK_ROOT / name).exists()]
if absent:
    raise FileNotFoundError(
        "Not found at /content: " + ", ".join(absent) + ".\n"
        "Drag them into the file browser in the left sidebar and wait for the upload to "
        "finish, or set USE_UPLOAD_WIDGET = True above. Check the names match exactly: a "
        "browser may have saved the archive as 'datasets (1).zip'."
    )

for name in REQUIRED_UPLOADS + ["train_manifest.csv"]:
    path = WORK_ROOT / name
    if path.exists():
        print(f"  {name}  ({path.stat().st_size / 1e6:.1f} MB)")

# Count files rather than test for the directory. A directory left behind by a failed
# extraction is empty, and treating its existence as success would skip the unzip forever.
already = sum(1 for p in EXTRACT_ROOT.rglob("*") if p.is_file()) if EXTRACT_ROOT.exists() else 0

if already:
    print(f"\n{EXTRACT_ROOT} already holds {already:,} files; skipping extraction. "
          "Delete the folder to force a fresh unzip.")
else:
    if EXTRACT_ROOT.exists():
        print(f"\n{EXTRACT_ROOT} exists but is empty; removing it and extracting again.")
        shutil.rmtree(EXTRACT_ROOT)
    print(f"\nExtracting {ARCHIVE_NAME}, this takes a couple of minutes ...")
    with zipfile.ZipFile(WORK_ROOT / ARCHIVE_NAME) as archive:
        archive.extractall(EXTRACT_ROOT)      # creates the directory itself
    print(f"Extracted {sum(1 for p in EXTRACT_ROOT.rglob('*') if p.is_file()):,} files.")

### 1.3 Discovering the Paths

The local edition takes its paths from a fixed repository layout. Nothing guarantees that layout
survives a zip, so this edition searches the extracted tree for the four things it needs and fails
with a readable message if any is absent.

`preprocessing.py` anchors its own paths to `Path(__file__).parent.parent`, which resolves
correctly inside the repository and incorrectly on a flat Colab session disk. Its module-level
constants are therefore reassigned to the discovered paths.

Reassigning the globals is necessary but not quite sufficient. `load_manifest` declares
`image_dir=TRAIN_IMAGE_DIR` as a default argument, and Python evaluates default arguments once at
definition time, so that one was already frozen to the module's own path before the reassignment
ran. It is rebound explicitly in Section 1.4. Every other function resolves its globals at call
time and needs nothing. The module file itself is left unmodified either way, so it stays
identical to the copy the other three task notebooks import.

In [ ]:
def find_path(root, name, kind):
    """Locate exactly one file or directory by name under root.

    Args:
        root: directory to search.
        name: exact file or directory name.
        kind: "dir" or "file".

    Returns:
        The single match, or raises with the candidates found.
    """
    matches = [p for p in root.rglob(name)
               if (p.is_dir() if kind == "dir" else p.is_file())]
    if not matches:
        raise FileNotFoundError(
            f"No {kind} named {name!r} under {root}. "
            f"Top level of the archive: {sorted(p.name for p in root.iterdir())}"
        )
    if len(matches) > 1:
        # A nested duplicate (a __MACOSX copy, or a doubly zipped folder) would otherwise be
        # picked arbitrarily and silently change which images are read.
        real = [p for p in matches if "__MACOSX" not in str(p)]
        if len(real) != 1:
            raise ValueError(f"Ambiguous: {len(matches)} matches for {name!r}: {matches[:5]}")
        matches = real
    return matches[0]


TRAIN_IMAGE_DIR = find_path(EXTRACT_ROOT, "images_train", "dir")
TEST_IMAGE_DIR = find_path(EXTRACT_ROOT, "images_test", "dir")
PREDICTION_TEMPLATE = find_path(EXTRACT_ROOT, "styles_prediction.csv", "file")

# The manifest is produced by notebook 00 and is not part of the archive, so it is looked for
# beside this notebook first and only then inside the extracted tree.
manifest_candidates = [WORK_ROOT / "train_manifest.csv"]
try:
    manifest_candidates.append(find_path(EXTRACT_ROOT, "train_manifest.csv", "file"))
except FileNotFoundError:
    pass
MANIFEST = next((p for p in manifest_candidates if p.exists()), None)

if MANIFEST is None:
    raise FileNotFoundError(
        "train_manifest.csv not found. It is written by Section 3.4 of "
        "00_eda_and_preprocessing.ipynb and is not part of A2_Fashion.zip. Run that notebook, "
        "then drag the manifest into /content alongside the archive."
    )

print("Manifest:      ", MANIFEST)
print("Train images:  ", TRAIN_IMAGE_DIR, f"({len(list(TRAIN_IMAGE_DIR.glob('*.jpg'))):,} files)")
print("Test images:   ", TEST_IMAGE_DIR, f"({len(list(TEST_IMAGE_DIR.glob('*.jpg'))):,} files)")
print("Template:      ", PREDICTION_TEMPLATE)

### 1.4 The Shared Preprocessing Module

`preprocessing.py` is imported rather than reimplemented, so this notebook and the other three
task notebooks apply byte-identical pixels. The reassignment below is the only Colab-specific
change and it touches paths only: no transform, no constant, and no function body is altered.

In [ ]:
if not (WORK_ROOT / "preprocessing.py").exists():
    raise FileNotFoundError(
        "preprocessing.py is not at /content. Drag it into the file browser in the left "
        "sidebar, at the top level, beside A2_Fashion.zip."
    )

sys.path.insert(0, str(WORK_ROOT))
import preprocessing

# Repoint the module at the discovered locations. Its own constants assume the repository
# layout, which a flat session disk does not reproduce. The functions resolve these globals at
# call time, so reassignment is sufficient.
preprocessing.MANIFEST = MANIFEST
preprocessing.TRAIN_IMAGE_DIR = TRAIN_IMAGE_DIR
preprocessing.TEST_IMAGE_DIR = TEST_IMAGE_DIR

# load_manifest declares `image_dir=TRAIN_IMAGE_DIR`, and Python evaluates default arguments
# once at definition time. The reassignment above therefore never reaches it: the default was
# frozen to the module's own path the moment the import ran. Rebind it explicitly. MANIFEST is
# read inside the function body, which is why the manifest itself loads without this.
import inspect

assert list(inspect.signature(preprocessing.load_manifest).parameters) == ["target", "image_dir"], (
    "load_manifest's signature has changed; the default rebinding below needs updating."
)
preprocessing.load_manifest.__defaults__ = (None, TRAIN_IMAGE_DIR)

from preprocessing import (
    IMAGE_TARGET_SIZE,
    compute_normalisation,
    describe_split,
    load_image_array,
    load_manifest,
    make_split,
)

# Prove the repointing worked before thirty cells depend on it.
_probe = load_manifest("articleType").head(3)
assert Path(_probe["path"].iloc[0]).exists(), (
    f"Manifest loaded but its first image path does not resolve: {_probe['path'].iloc[0]}"
)
print(f"Module wired up. Manifest rows with an articleType label: "
      f"{len(load_manifest('articleType')):,}")
print("Image target size (w, h):", IMAGE_TARGET_SIZE)

### 1.5 Configuration

Every setting the notebook reuses is defined once here. Two rules govern this cell.

- **Nothing below re-declares a preprocessing constant.** `IMAGE_TARGET_SIZE` comes from `preprocessing`, so this notebook cannot silently disagree with the audit that produced the manifest. The data paths come from Section 1.3 and are not repeated here either.
- **Every hyperparameter that Section 7 tunes is named here**, so a tuning run changes one cell rather than hunting through the training code.

The defaults are the starting point, not the tuned result. Section 10 lists what to sweep and in what order.

#### Performance Notes

Three settings above are about throughput and memory rather than modelling, and none of them
changes what is learned.

- **`ALLOW_CPU = False`** stops the notebook if no GPU is visible. A silent CPU fallback is the
  worst failure mode available here: the ResNet is roughly a hundred times slower on a CPU, which
  presents as a hang rather than an error, and the host-side activations at batch 128 are large
  enough to have the operating system kill the kernel without a traceback.
- **`USE_AMP`** runs the forward and backward passes in bf16 where the card supports it and fp16
  with a gradient scaler otherwise. Parameters, the optimiser state and every reported metric stay
  in fp32.
- **`CACHE_ON_DEVICE`** holds the decoded `uint8` images in VRAM, about 0.6 GB. It removes the same
  amount from host RAM and eliminates the per-batch host-to-device copy.

Expect roughly a 3-5x speedup over the original CPU-side pipeline on a mid-range GPU, and a host
footprint under 1 GB after Section 3 releases the HOG features.

In [ ]:
TARGET = "articleType"
RANDOM_STATE = 42

# --- Runtime controls -------------------------------------------------------------
QUICK_RUN = False        # True: 3 epochs, 1 seed, 5,000 training rows. Structural check only.
RUN_SEED_STUDY = True    # False: skip the multi-seed variance runs (the most expensive section)
RUN_SALIENCY = True      # False: skip the gradient saliency figures

# --- Performance ------------------------------------------------------------------
# ALLOW_CPU exists to make a silent CPU fallback impossible. Training this ResNet on a CPU is
# roughly a hundred times slower than on a mid-range GPU, which presents as a hang rather than
# as an error, so the notebook refuses to start instead. Set it True only for a QUICK_RUN.
ALLOW_CPU = False
USE_AMP = True           # mixed precision on CUDA. bf16 where supported, else fp16 with a scaler
CHANNELS_LAST = True     # NHWC layout, which is what the tensor-core convolution kernels want
CACHE_ON_DEVICE = True   # hold the uint8 image cache in VRAM: ~0.6 GB, and it frees host RAM

# --- Split ------------------------------------------------------------------------
VALIDATION_SHARE = 0.20

# --- Optimisation -----------------------------------------------------------------
BATCH_SIZE = 128
EPOCHS = 40
PATIENCE = 8             # early stopping, measured on validation macro-F1
LEARNING_RATE = 1e-3
WEIGHT_DECAY = 1e-4
WARMUP_EPOCHS = 3
LABEL_SMOOTHING = 0.05   # justified by the documented label noise, Section 3.2.2 of notebook 00

# --- Augmentation -----------------------------------------------------------------
AUG_FLIP_PROBABILITY = 0.5
AUG_ROTATION_DEGREES = 10.0
AUG_TRANSLATE_FRACTION = 0.08
AUG_JITTER_STRENGTH = 0.2

# --- Long-tail treatments ---------------------------------------------------------
LOGIT_ADJUST_TAU = 1.0   # Menon et al. (2021); 0 disables the adjustment
STAGE2_EPOCHS = 10       # decoupled classifier retraining, Kang et al. (2020)
STAGE2_LR = 1e-2

# --- Seed variance ----------------------------------------------------------------
SEEDS = [42, 1337, 2024]

# --- Support buckets for the tail-aware metric breakdown --------------------------
SUPPORT_BUCKETS = [
    ("head (>=1000)", 1000, np.inf),
    ("body (100-999)", 100, 1000),
    ("tail (10-99)", 10, 100),
    ("rare (<10)", 0, 10),
]

# --- Output locations -------------------------------------------------------------
# Session disk. Section 9.1 downloads both; anything left only under /content is lost when
# the runtime recycles. PREDICTION_TEMPLATE was discovered in Section 1.3.
ARTEFACT_DIR = WORK_ROOT / "models" / "task1"
OUTPUT_DIR = WORK_ROOT / "outputs"

if QUICK_RUN:
    EPOCHS, STAGE2_EPOCHS, WARMUP_EPOCHS = 3, 2, 1
    SEEDS = [RANDOM_STATE]
    RUN_SEED_STUDY = False

print("QUICK_RUN:", QUICK_RUN, "| epochs:", EPOCHS, "| seeds:", SEEDS)

In [ ]:
def set_seed(seed):
    """Seed every generator this notebook draws from.

    Torch, NumPy and Python are all seeded because the augmentation, the weight
    initialisation and the batch order each draw from a different one. Without all three,
    a "same seed" rerun is not actually a rerun.
    """
    import random
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)


DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

if DEVICE.type == "cuda":
    name = torch.cuda.get_device_name(0)
    total = torch.cuda.get_device_properties(0).total_memory / 1e9
    capability = torch.cuda.get_device_capability(0)
    print(f"GPU: {name} | {total:.1f} GB | compute capability {capability[0]}.{capability[1]}")
    print(f"torch {torch.__version__} built against CUDA {torch.version.cuda}")

    # A wheel without kernels for this card fails here rather than mid-epoch, and the failure
    # is a clear one instead of a process that disappears without a traceback.
    try:
        _ = (torch.zeros(8, device=DEVICE) + 1).sum().item()
    except RuntimeError as error:
        raise RuntimeError(
            f"CUDA is visible but cannot execute a kernel on this card: {error}\n"
            "This is the usual symptom of a PyTorch build without kernels for your GPU. "
            "RTX 50-series cards need a CUDA 12.8 build:\n"
            "  Runtime > Change runtime type > T4 GPU, then rerun from Section 1.1."
        ) from error

    torch.backends.cudnn.benchmark = True          # fixed input size, so autotuning pays off
    torch.backends.cuda.matmul.allow_tf32 = True
    torch.backends.cudnn.allow_tf32 = True
elif not ALLOW_CPU:
    raise RuntimeError(
        "No CUDA device. Training the ResNet on a CPU takes roughly 10 minutes per epoch "
        "per thousand images, which looks like a hang rather than an error.\n"
        "Set Runtime > Change runtime type > T4 GPU and rerun, or set ALLOW_CPU = True in "
        "Section 1.5 and QUICK_RUN = True to accept the slowdown deliberately."
    )
else:
    print("Running on CPU by explicit request (ALLOW_CPU = True). Expect this to be slow.")

# bf16 has the dynamic range of fp32 and needs no loss scaling. Turing cards such as the T4
# lack it, so fp16 with a gradient scaler is the fallback. Neither changes what is learned.
AMP_ENABLED = USE_AMP and DEVICE.type == "cuda"
AMP_DTYPE = (torch.bfloat16 if AMP_ENABLED and torch.cuda.is_bf16_supported() else torch.float16)
CHANNELS_LAST = CHANNELS_LAST and DEVICE.type == "cuda"
CACHE_ON_DEVICE = CACHE_ON_DEVICE and DEVICE.type == "cuda"

print(f"Mixed precision: {AMP_ENABLED} ({str(AMP_DTYPE).replace('torch.', '') if AMP_ENABLED else 'fp32'})"
      f" | channels_last: {CHANNELS_LAST} | image cache on device: {CACHE_ON_DEVICE}")

set_seed(RANDOM_STATE)
print("Seeded at", RANDOM_STATE)

## 2. Data and the Evaluation Framework

This section is built first and then frozen. Nothing after it may change the split, the metric
definitions, or the baselines, because a comparison table is only meaningful if every row was
scored the same way on the same rows.

### 2.1 The Split

`make_split` implements rules 1, 3 and 4 of the handover in Section 5 of notebook 00: whole
`group_id` values stay on one side so byte-identical images cannot leak across, the draw is
stratified on the label, and classes with fewer than two groups go entirely to training because
a class with one example cannot also be evaluated.

The consequence worth stating up front is the **classes absent from validation** count printed
below. Those classes are trainable but unscoreable. Every macro average in this notebook is
therefore computed over the classes present in validation, and the absent count is reported
beside it so the denominator is never ambiguous.

In [ ]:
frame = load_manifest(TARGET)
print(f"Rows carrying an {TARGET} label: {len(frame):,}")
print(f"Distinct classes in the manifest: {frame[TARGET].nunique()}")

train_frame, val_frame = make_split(
    frame, TARGET, validation_share=VALIDATION_SHARE, random_state=RANDOM_STATE
)

if QUICK_RUN:
    # Stratified where possible; a plain sample is enough for a structural check.
    train_frame = train_frame.sample(n=min(5000, len(train_frame)), random_state=RANDOM_STATE)
    train_frame = train_frame.reset_index(drop=True)
    print("QUICK_RUN: training rows reduced to", len(train_frame))

display(describe_split(train_frame, val_frame, TARGET))

In [ ]:
# Label encoding. Fixed to the sorted training classes and persisted in Section 9, because
# reconstructing it later from a different frame would silently permute every prediction.
CLASSES = sorted(train_frame[TARGET].unique())

if QUICK_RUN:
    # The subsample above can drop whole classes, which would leave validation rows with no
    # index to map to. Full runs never enter this branch: make_split keeps every class in training.
    keep = val_frame[TARGET].isin(CLASSES)
    print(f"QUICK_RUN: dropping {int((~keep).sum())} validation rows whose class was "
          "removed by the training subsample.")
    val_frame = val_frame.loc[keep].reset_index(drop=True)

CLASS_TO_INDEX = {label: index for index, label in enumerate(CLASSES)}
N_CLASSES = len(CLASSES)

# Any validation class absent from training cannot be predicted. make_split sends
# single-group classes to training, so this should be empty; the check is what proves it.
unseen = sorted(set(val_frame[TARGET]) - set(CLASSES))
assert not unseen, f"Validation holds classes never seen in training: {unseen}"

y_train = train_frame[TARGET].map(CLASS_TO_INDEX).to_numpy()
y_val = val_frame[TARGET].map(CLASS_TO_INDEX).to_numpy()

train_support = pd.Series(np.bincount(y_train, minlength=N_CLASSES), index=CLASSES)
val_support = pd.Series(np.bincount(y_val, minlength=N_CLASSES), index=CLASSES)
SCOREABLE = np.flatnonzero(val_support.to_numpy() > 0)   # class indices macro averages use

print(f"Classes: {N_CLASSES}")
print(f"Scoreable in validation: {len(SCOREABLE)} | absent: {N_CLASSES - len(SCOREABLE)}")
print(f"Training support range: {train_support.max():,} down to {train_support.min()}")
print("Absent from validation:", sorted(np.array(CLASSES)[val_support.to_numpy() == 0]))

In [ ]:
# Support buckets, assigned from TRAINING support so the grouping is a property of the
# learning problem rather than of the particular validation draw.
def assign_bucket(count):
    for name, low, high in SUPPORT_BUCKETS:
        if low <= count < high:
            return name
    return SUPPORT_BUCKETS[-1][0]


BUCKET_OF_CLASS = train_support.map(assign_bucket)
BUCKET_INDICES = {
    name: np.array([CLASS_TO_INDEX[c] for c in BUCKET_OF_CLASS.index[BUCKET_OF_CLASS == name]])
    for name, _, _ in SUPPORT_BUCKETS
}

bucket_table = pd.DataFrame([
    {
        "Bucket": name,
        "Classes": len(indices),
        "Scoreable in validation": len(np.intersect1d(indices, SCOREABLE)),
        "Training images": int(train_support.to_numpy()[indices].sum()) if len(indices) else 0,
        "Share of training images %": (
            train_support.to_numpy()[indices].sum() / len(y_train) * 100 if len(indices) else 0.0
        ),
    }
    for name, indices in BUCKET_INDICES.items()
])
display(bucket_table.style.format({"Share of training images %": "{:.1f}%"}))

#### Why the Buckets Matter More Than the Headline

The table above is the reason a single macro-F1 is not enough on its own. A handful of head
classes account for most of the images while most of the classes hold a small fraction of them,
so two models with the same macro-F1 can have completely different failure profiles: one may be
competent everywhere and mediocre on the head, the other excellent on the head and near zero on
the tail. Reporting macro-F1 per bucket separates those two cases, and it is the breakdown that
Section 8 uses to choose between the finalists.

### 2.2 Normalisation Constants

Rule 5 of the handover: fitted on the training rows only, then applied unchanged to validation
and to the test set. The constants are persisted in Section 9 because inference must reproduce
them exactly; recomputing them at prediction time on a different population would shift the
input distribution the model was trained for.

In [ ]:
start = time.time()
NORM_MEAN, NORM_STD = compute_normalisation(train_frame, target_size=IMAGE_TARGET_SIZE)
print(f"Fitted on {len(train_frame):,} training rows in {time.time() - start:.0f}s")
print("Mean (R, G, B):", np.round(NORM_MEAN, 4))
print("Std  (R, G, B):", np.round(NORM_STD, 4))

# White studio backgrounds dominate, so a mean near 0.9 is the expected result rather than a bug.
assert (NORM_MEAN > 0.5).all(), "Unexpectedly dark mean; check the transform before continuing."

### 2.3 Loading the Images into Memory

The whole training set is 60x80x3 bytes per image, so all 37,846 images occupy roughly 545 MB as
`uint8`. Holding them in RAM and decoding once is worth doing: it removes JPEG decoding from every
epoch, which on images this small is otherwise the dominant cost and would leave the GPU idle.

Augmentation still happens per epoch on tensors, so nothing about the training distribution is
frozen by this cache. Only the deterministic transform from Section 3.1 of notebook 00 is applied
here, exactly once per image.

In [ ]:
def build_image_cache(frame, description):
    """Decode a frame's images once through the shared transform into one uint8 array.

    Only the deterministic transform from Section 3.1 of notebook 00 is applied here, exactly
    once per image. Augmentation still happens per epoch, so nothing about the training
    distribution is frozen by this cache.

    Returns:
        Array of shape (rows, height, width, 3), dtype uint8, in the frame's row order.
    """
    width, height = IMAGE_TARGET_SIZE
    images = np.empty((len(frame), height, width, 3), dtype=np.uint8)
    start = time.time()
    for position, path in enumerate(frame["path"]):
        images[position] = load_image_array(path, target_size=IMAGE_TARGET_SIZE, scale=False)
        if position and position % 10000 == 0:
            print(f"  {description}: {position:,} / {len(frame):,}")
    print(f"{description}: {len(frame):,} images in {time.time() - start:.0f}s "
          f"({images.nbytes / 1e6:.0f} MB)")
    return images


def report_memory(label=""):
    """Host RSS and, on CUDA, device allocation. Cheap, and it makes a leak visible early."""
    line = []
    try:
        import resource
        peak_kb = resource.getrusage(resource.RUSAGE_SELF).ru_maxrss
        line.append(f"host peak {peak_kb / 1e6:.2f} GB")
    except (ImportError, AttributeError):
        try:
            import psutil
            line.append(f"host RSS {psutil.Process().memory_info().rss / 1e9:.2f} GB")
        except ImportError:
            pass
    if DEVICE.type == "cuda":
        line.append(f"device allocated {torch.cuda.memory_allocated() / 1e9:.2f} GB "
                    f"reserved {torch.cuda.memory_reserved() / 1e9:.2f} GB")
    print(f"[memory{' ' + label if label else ''}] " + " | ".join(line))


X_train_images = build_image_cache(train_frame, "train")
X_val_images = build_image_cache(val_frame, "validation")

assert len(X_train_images) == len(y_train) and len(X_val_images) == len(y_val)
report_memory("after caching")

### 2.4 Augmentation Policy and Batching

The policy is unchanged from the original notebook. Each choice follows from a finding in
notebook 00 rather than from a default recipe.

| Transform | Justification |
|---|---|
| Horizontal flip, p=0.5 | Catalogue shots have no meaningful left-right orientation, so a flip produces a plausible product photograph. |
| Rotation +/- 10 degrees, translate +/- 8%, fill white | Simulates the small framing variation the catalogue already contains. White fill matches `IMAGE_PAD_RGB`, so an augmented border is indistinguishable from the padding the transform itself applies. |
| Colour jitter 0.2 | `baseColour` is not the target and is excluded from inputs, so colour invariance is desirable: it discourages the model from using colour as a shortcut for article type. |
| **No vertical flip** | Upside-down garments do not occur; the augmentation would move the training distribution away from the test distribution. |
| **No random crop** | Section 3.1 of notebook 00 argues the portrait frame carries class cues at both the top and bottom of a product. A crop deletes exactly the evidence that separates similar article types. |

#### What Changed, and Why It Does Not Change the Policy

The original implementation used `torchvision.transforms` inside a `Dataset`, which applies each
transform to one image at a time on the CPU. At 60x80 the model is small enough that this made
the CPU the bottleneck: the GPU spent most of each epoch waiting for batches, and every image
passed through the host as a separate Python-level call.

This version applies the identical transforms to a whole batch on the device, with **per-sample
random parameters** so that augmentation diversity is exactly what it was before. Two details are
worth stating because they are the only places the implementations differ:

- **Rotation is corrected for the aspect ratio.** `affine_grid` works in normalised coordinates,
  where a naive rotation matrix shears a non-square image. The matrix below rescales by 80/60 so
  the rotation is a true rotation in pixel space, which is what `torchvision` does.
- **White fill is obtained by offsetting.** `grid_sample` can only pad with zeros, so the batch is
  shifted by -1, sampled, and shifted back, which makes out-of-frame pixels exactly 1.0, that is
  white. This is identical to `fill=1.0`, not an approximation of it.

Colour jitter applies brightness, then saturation, then contrast in a fixed order rather than the
shuffled order `torchvision` uses. With independent per-sample factors drawn from the same range,
the resulting distribution of images is equivalent for this purpose.

The `uint8` cache is uploaded to the device once. It costs about 0.6 GB of VRAM and removes the
same amount from host RAM, along with the per-batch host-to-device transfer.

In [ ]:
LUMA = torch.tensor([0.299, 0.587, 0.114], device=DEVICE).view(1, 3, 1, 1)
NORM_MEAN_T = torch.tensor(NORM_MEAN, dtype=torch.float32, device=DEVICE).view(1, 3, 1, 1)
NORM_STD_T = torch.tensor(NORM_STD, dtype=torch.float32, device=DEVICE).view(1, 3, 1, 1)


def augment_batch(x):
    """Apply the Section 2.4 policy to a batch, with independent parameters per sample.

    Args:
        x: float tensor of shape (n, 3, height, width) with values in [0, 1].

    Returns:
        A tensor of the same shape and range.
    """
    n = x.shape[0]
    device = x.device

    # Horizontal flip. torch.where selects per sample, so the draw is genuinely per image.
    flip = torch.rand(n, device=device) < AUG_FLIP_PROBABILITY
    x = torch.where(flip.view(-1, 1, 1, 1), x.flip(-1), x)

    # Rotation and translation in one affine warp. The height/width factors correct for
    # affine_grid's normalised coordinates, without which the rotation would shear.
    height, width = x.shape[-2], x.shape[-1]
    angle = (torch.rand(n, device=device) * 2 - 1) * math.radians(AUG_ROTATION_DEGREES)
    shift_x = (torch.rand(n, device=device) * 2 - 1) * AUG_TRANSLATE_FRACTION * 2
    shift_y = (torch.rand(n, device=device) * 2 - 1) * AUG_TRANSLATE_FRACTION * 2
    cos, sin = torch.cos(angle), torch.sin(angle)

    theta = torch.zeros(n, 2, 3, device=device)
    theta[:, 0, 0] = cos
    theta[:, 0, 1] = -sin * height / width
    theta[:, 0, 2] = shift_x
    theta[:, 1, 0] = sin * width / height
    theta[:, 1, 1] = cos
    theta[:, 1, 2] = shift_y

    grid = F.affine_grid(theta, list(x.shape), align_corners=False)
    # Offset by -1 so that grid_sample's zero padding lands on white once shifted back.
    x = F.grid_sample(x - 1.0, grid, mode="bilinear", padding_mode="zeros",
                      align_corners=False) + 1.0

    # Colour jitter: brightness, saturation, contrast, each with its own per-sample factor.
    def factor():
        return 1.0 + (torch.rand(n, 1, 1, 1, device=device) * 2 - 1) * AUG_JITTER_STRENGTH

    x = x * factor()
    grey = (x * LUMA).sum(dim=1, keepdim=True)
    x = (x - grey) * factor() + grey
    mean = grey.mean(dim=(2, 3), keepdim=True)
    x = (x - mean) * factor() + mean

    return x.clamp_(0.0, 1.0)


class BatchStream:
    """Device-resident batches, replacing Dataset plus DataLoader.

    The images are held once as uint8 in the device's memory and converted to float per batch,
    so nothing is copied from the host during training and the host holds no per-batch buffers.

    Args:
        images: uint8 array of shape (rows, height, width, 3), or an existing device tensor
            to share with another stream.
        labels: integer class indices aligned to `images`.
        batch_size: rows per batch.
        augment: apply the Section 2.4 policy. Training only.
        shuffle: reorder each epoch. Ignored when `weights` is given.
        weights: per-row sampling weights for class-balanced draws with replacement. Used by
            the decoupled stage in Section 6.2.
    """

    def __init__(self, images, labels, batch_size=None, augment=False, shuffle=False,
                 weights=None):
        if torch.is_tensor(images):
            self.images = images                      # shared with another stream, not copied
        else:
            self.images = torch.from_numpy(np.ascontiguousarray(images))
            if CACHE_ON_DEVICE:
                self.images = self.images.to(DEVICE, non_blocking=True)
        self.labels = torch.as_tensor(np.asarray(labels), dtype=torch.long, device=DEVICE)
        self.batch_size = batch_size or BATCH_SIZE
        self.augment = augment
        self.shuffle = shuffle
        if weights is None:
            self.weights = None
        elif torch.is_tensor(weights):
            self.weights = weights.to(device=self.images.device, dtype=torch.double)
        else:
            self.weights = torch.as_tensor(np.asarray(weights), dtype=torch.double,
                                           device=self.images.device)

    def variant(self, **overrides):
        """A stream over the same device tensor with different batching or sampling."""
        settings = {"batch_size": self.batch_size, "augment": self.augment,
                    "shuffle": self.shuffle, "weights": self.weights}
        settings.update(overrides)
        return BatchStream(self.images, self.labels.cpu().numpy(), **settings)

    def __len__(self):
        return math.ceil(len(self.labels) / self.batch_size)

    def _order(self):
        n = len(self.labels)
        if self.weights is not None:
            return torch.multinomial(self.weights, n, replacement=True)
        if self.shuffle:
            return torch.randperm(n, device=self.images.device)
        return torch.arange(n, device=self.images.device)

    def __iter__(self):
        order = self._order()
        for start in range(0, len(order), self.batch_size):
            index = order[start:start + self.batch_size]
            batch = self.images[index].to(DEVICE, non_blocking=True)
            x = batch.permute(0, 3, 1, 2).float().div_(255.0)
            if self.augment:
                x = augment_batch(x)
            x = (x - NORM_MEAN_T) / NORM_STD_T
            if CHANNELS_LAST:
                x = x.contiguous(memory_format=torch.channels_last)
            yield x, self.labels[index.to(self.labels.device)]


def make_loaders(batch_size=BATCH_SIZE, weights=None):
    """Training and validation streams. Reuses the uploaded tensors; nothing is re-copied."""
    return (train_loader.variant(batch_size=batch_size, weights=weights,
                                 shuffle=weights is None),
            val_loader)


train_loader = BatchStream(X_train_images, y_train, BATCH_SIZE, augment=True, shuffle=True)
val_loader = BatchStream(X_val_images, y_val, 512, augment=False, shuffle=False)

batch_images, batch_labels = next(iter(train_loader))
print("Batch tensor:", tuple(batch_images.shape), batch_images.dtype, batch_images.device)
print(f"Normalised range: [{batch_images.min():.2f}, {batch_images.max():.2f}]")
print(f"Batches per epoch: {len(train_loader)}")
report_memory("after upload")

In [ ]:
# Look at what the model actually receives. An augmentation bug is far cheaper to catch here
# than to diagnose from a training curve.
def to_displayable(x01):
    """A [0, 1] CHW tensor as an HWC array ready for imshow."""
    return x01.detach().float().clamp(0, 1).permute(1, 2, 0).cpu().numpy()


sample_positions = np.random.RandomState(RANDOM_STATE).choice(len(y_train), 6, replace=False)
index = torch.as_tensor(sample_positions, device=train_loader.images.device)
raw = train_loader.images[index].to(DEVICE).permute(0, 3, 1, 2).float().div(255.0)

set_seed(RANDOM_STATE)
augmented = augment_batch(raw.clone())

fig, axes = plt.subplots(2, 6, figsize=(11, 4.4))
for column in range(6):
    axes[0, column].imshow(to_displayable(raw[column]))
    axes[0, column].set_title(CLASSES[y_train[sample_positions[column]]], fontsize=7)
    axes[1, column].imshow(to_displayable(augmented[column]))
    for row in range(2):
        axes[row, column].axis("off")
fig.suptitle("Deterministic transform (top) and one augmented draw (bottom)", y=1.02)
plt.tight_layout(); plt.show()

# White padding must survive the warp: an augmented border darker than the studio background
# would be a fill colour bug, and would teach the model a border cue that the test set lacks.
corners = augmented[:, :, :2, :2]
print(f"Corner pixels after augmentation: min {corners.min():.3f}, mean {corners.mean():.3f} "
      "(1.0 is white)")

### 2.5 Metrics

One function scores every model in this notebook. Five headline numbers plus the bucket
breakdown, each answering a different question.

| Metric | Question it answers |
|---|---|
| Top-1 accuracy | How often is the system exactly right? Dominated by the head, and comparable to published work. |
| **Macro-F1** | How well does it do averaged over classes rather than images? The primary metric, because every article type matters to a catalogue regardless of stock volume. |
| Balanced accuracy | Macro-recall. Read beside macro-F1 it separates "misses tail classes" from "over-predicts tail classes". |
| Weighted F1 | Macro-F1's image-weighted counterpart, for readers who care about aggregate throughput. |
| Top-5 accuracy | The realistic figure for a tagging assistant that proposes candidates for a human to confirm. |

All macro averages are taken over `SCOREABLE`, the classes with at least one validation example.

In [ ]:
def evaluate_predictions(y_true, y_pred, scores=None, name=""):
    """Score one model's validation predictions.

    Args:
        y_true: integer class indices.
        y_pred: integer class indices.
        scores: optional (rows, N_CLASSES) array of logits, probabilities, or decision
            values. Only the ranking is used, so any monotone score works for top-5.
        name: row label in the results table.

    Returns:
        A one-row DataFrame.
    """
    row = {
        "Model": name,
        "Top-1 accuracy": accuracy_score(y_true, y_pred),
        "Macro-F1": f1_score(y_true, y_pred, labels=SCOREABLE, average="macro", zero_division=0),
        "Balanced accuracy": balanced_accuracy_score(y_true, y_pred),
        "Weighted F1": f1_score(y_true, y_pred, average="weighted", zero_division=0),
    }

    if scores is not None:
        top5 = np.argsort(scores, axis=1)[:, -5:]
        row["Top-5 accuracy"] = float(np.mean([t in row5 for t, row5 in zip(y_true, top5)]))
    else:
        row["Top-5 accuracy"] = np.nan

    per_class = f1_score(
        y_true, y_pred, labels=np.arange(N_CLASSES), average=None, zero_division=0
    )
    for bucket, indices in BUCKET_INDICES.items():
        scoreable_in_bucket = np.intersect1d(indices, SCOREABLE)
        row[f"F1 {bucket}"] = (
            float(per_class[scoreable_in_bucket].mean()) if len(scoreable_in_bucket) else np.nan
        )

    return pd.DataFrame([row])


def per_class_f1(y_true, y_pred):
    """Per-class F1 as a Series indexed by class name, restricted to scoreable classes."""
    scores = f1_score(y_true, y_pred, labels=np.arange(N_CLASSES), average=None, zero_division=0)
    return pd.Series(scores, index=CLASSES).iloc[SCOREABLE]


RESULTS = []   # every scored model appends its one-row frame here


def record(result_frame):
    RESULTS.append(result_frame)
    display(result_frame.style.format({c: "{:.4f}" for c in result_frame.columns if c != "Model"}))
    return result_frame

### 2.6 Baselines

Two, and neither is a strawman.

- **Majority class** always predicts the most frequent training class. Its accuracy is the number
  any model must beat before its accuracy means anything, and its macro-F1 near zero is the
  concrete demonstration that accuracy is the wrong headline for this target.
- **Stratified random** samples from the training class prior. It is the floor for macro-F1
  specifically: a model that scores below it has learned nothing transferable about the tail.

In [ ]:
majority_index = int(np.bincount(y_train, minlength=N_CLASSES).argmax())
majority_pred = np.full_like(y_val, majority_index)
majority_scores = np.zeros((len(y_val), N_CLASSES))
majority_scores[:, majority_index] = 1.0
print(f"Majority class: {CLASSES[majority_index]} "
      f"({train_support.iloc[majority_index]:,} training images)")
record(evaluate_predictions(y_val, majority_pred, majority_scores, "Baseline: majority class"))

prior = np.bincount(y_train, minlength=N_CLASSES) / len(y_train)
rng = np.random.RandomState(RANDOM_STATE)
stratified_pred = rng.choice(N_CLASSES, size=len(y_val), p=prior)
record(evaluate_predictions(
    y_val, stratified_pred, np.tile(prior, (len(y_val), 1)), "Baseline: stratified random"
))

#### Reading the Baselines

The majority-class row is the argument for the whole metric framework. Its top-1 accuracy is
substantial while its macro-F1 is effectively zero, which is exactly the gap a headline accuracy
would hide. Any claim later in this notebook that a model "performs well" is a claim about the
macro column and the bucket breakdown, not about the accuracy column.

## 3. Model 1: HOG + Linear SVM

The baseline model, and a deliberate one rather than a formality.

Section 2.3 of notebook 00 showed that article types separate largely by **silhouette**: the class
mean images that came out most similar were the ones sharing an outline against the white studio
background. A Histogram of Oriented Gradients descriptor encodes precisely that, local edge
orientation, and discards the colour and texture detail that a 60x80 JPEG barely carries anyway.
It is therefore the strongest classical representation available for this data, not a strawman.

What this model establishes: whether the task needs a learned representation at all. If a linear
classifier over hand-designed edge features closes most of the gap, the deep models must justify
their cost. If it does not, the gap is the evidence for the CNN.

The descriptor uses 8x8 pixel cells over the 60x80 frame with 2x2 block normalisation, giving a
6x9 block grid and 1,944 features per image.

In [ ]:
from skimage.feature import hog

HOG_PARAMS = dict(
    orientations=9,
    pixels_per_cell=(8, 8),
    cells_per_block=(2, 2),
    block_norm="L2-Hys",
    feature_vector=True,
)


def hog_features(images, description):
    """HOG descriptor per image, computed on the luminance channel.

    One image at a time, in float32, writing straight into a preallocated output. The obvious
    vectorised form promotes the whole uint8 cache to float64, which peaks at several GB for
    no speed gain and is the single largest host allocation in the notebook.

    Args:
        images: uint8 array of shape (rows, height, width, 3).

    Returns:
        float32 array of shape (rows, n_features).
    """
    start = time.time()
    weights = np.array([0.299, 0.587, 0.114], dtype=np.float32)
    features = None
    for position, image in enumerate(images):
        grey = (image.astype(np.float32) @ weights) / 255.0
        descriptor = hog(grey, **HOG_PARAMS).astype(np.float32)
        if features is None:
            features = np.empty((len(images), descriptor.size), dtype=np.float32)
        features[position] = descriptor
    print(f"{description}: {features.shape[0]:,} x {features.shape[1]} features "
          f"in {time.time() - start:.0f}s ({features.nbytes / 1e6:.0f} MB)")
    return features


hog_train = hog_features(X_train_images, "train HOG")
hog_val = hog_features(X_val_images, "validation HOG")
report_memory("after HOG")

In [ ]:
# dual=False because n_samples greatly exceeds n_features, which is the regime the primal
# solver is built for. class_weight='balanced' is the classical counterpart of the
# reweighting the deep models test in Section 6, so the comparison stays like for like.
start = time.time()
svm = LinearSVC(C=0.1, dual=False, class_weight="balanced", max_iter=3000,
                random_state=RANDOM_STATE)
svm.fit(hog_train, y_train)
print(f"Fitted in {time.time() - start:.0f}s")

svm_scores = svm.decision_function(hog_val)
svm_pred = svm_scores.argmax(axis=1)
record(evaluate_predictions(y_val, svm_pred, svm_scores, "1. HOG + linear SVM"))

# liblinear holds a float64 copy of the training matrix internally, so these arrays and the
# uint8 caches together are roughly a gigabyte that nothing after this section reads: the
# streams already hold their own copies of the images. Releasing it here is what keeps the
# host footprint flat for the rest of the run.
del hog_train, hog_val
if CACHE_ON_DEVICE:
    del X_train_images, X_val_images
gc.collect()
report_memory("after releasing HOG")

#### What the Baseline Establishes

Read this row against the two baselines above and against the CNN that follows. Three outcomes are
possible and each carries a different conclusion for the report:

- The SVM beats the majority baseline substantially on macro-F1 but is well short of the CNN.
  Silhouette carries real signal, and the CNN's advantage is a learned representation rather than
  a trivially better fit.
- The SVM is close to the CNN. The deep model is not earning its training cost, and the honest
  recommendation would be the cheaper classical pipeline.
- The SVM collapses on the tail buckets while holding the head. Hand-designed features generalise
  from many examples but cannot form a class from five, which is the specific weakness the
  advanced model in Section 6 is designed to address.

The bucket columns are what distinguish these cases; the headline accuracy does not.

## 4. Training Machinery

Written once and shared by both neural models, so that any difference in their results comes from
the architecture or the objective rather than from an accidentally different training recipe.

Three choices worth stating.

- **Early stopping on validation macro-F1, not on validation loss.** Cross-entropy on this
  distribution is dominated by head classes, so the epoch that minimises loss is not the epoch
  that best serves the tail. Selecting on the primary metric keeps model selection aligned with
  the evaluation framework.
- **Cosine schedule with a linear warmup.** Warmup stabilises the first epochs at batch 128 with
  BatchNorm; the cosine decay removes a step-size hyperparameter that would otherwise need tuning.
- **The best epoch's weights are restored**, not the last epoch's, so the reported score and the
  saved checkpoint are the same model.

In [ ]:
def prepare_model(model):
    """Move a model to the device in the memory layout the convolution kernels prefer."""
    model = model.to(DEVICE)
    if CHANNELS_LAST:
        model = model.to(memory_format=torch.channels_last)
    return model


def run_epoch(model, loader, criterion, optimiser=None, logit_bias=None, scaler=None):
    """One pass over a stream. Trains if an optimiser is given, otherwise evaluates.

    Args:
        logit_bias: optional (N_CLASSES,) tensor added to the logits before the loss.
            Used by logit-adjusted cross-entropy in Section 6; never applied at evaluation,
            because the adjustment belongs to the training objective and not to the model.
        scaler: gradient scaler, required only for fp16. bf16 needs none.

    Returns:
        (mean loss, logits array or None). Logits are returned in float32 whatever the
        autocast dtype was, so every metric downstream sees the same precision as before.
    """
    training = optimiser is not None
    model.train(training)
    total_loss, n_seen, collected = 0.0, 0, []

    with torch.set_grad_enabled(training):
        for images, labels in loader:
            with torch.autocast(device_type=DEVICE.type, dtype=AMP_DTYPE, enabled=AMP_ENABLED):
                logits = model(images)
                loss = criterion(logits if logit_bias is None else logits + logit_bias, labels)

            if training:
                optimiser.zero_grad(set_to_none=True)
                if scaler is not None and scaler.is_enabled():
                    scaler.scale(loss).backward()
                    scaler.step(optimiser)
                    scaler.update()
                else:
                    loss.backward()
                    optimiser.step()
            else:
                collected.append(logits.detach().float().cpu())

            total_loss += loss.item() * len(labels)
            n_seen += len(labels)

    logits_out = torch.cat(collected).numpy() if collected else None
    return total_loss / n_seen, logits_out


def make_scaler():
    """A gradient scaler, enabled only for fp16. bf16 has fp32's range and needs no scaling."""
    enabled = AMP_ENABLED and AMP_DTYPE == torch.float16
    try:
        return torch.amp.GradScaler(DEVICE.type, enabled=enabled)
    except (AttributeError, TypeError):      # torch < 2.4 keeps it under torch.cuda.amp
        return torch.cuda.amp.GradScaler(enabled=enabled)


def fit(model, train_loader, val_loader, epochs=EPOCHS, lr=LEARNING_RATE,
        weight_decay=WEIGHT_DECAY, logit_bias=None, patience=PATIENCE,
        parameters=None, label="", verbose_every=1):
    """Train with warmup + cosine decay, early stopping on validation macro-F1.

    Returns:
        (history DataFrame, best validation logits, best macro-F1).
    """
    model = prepare_model(model)
    criterion = nn.CrossEntropyLoss(label_smoothing=LABEL_SMOOTHING)
    optimiser = torch.optim.AdamW(
        parameters if parameters is not None else model.parameters(),
        lr=lr, weight_decay=weight_decay,
    )
    scaler = make_scaler()

    def schedule(epoch):
        if epoch < WARMUP_EPOCHS:
            return (epoch + 1) / max(WARMUP_EPOCHS, 1)
        progress = (epoch - WARMUP_EPOCHS) / max(epochs - WARMUP_EPOCHS, 1)
        return 0.5 * (1 + np.cos(np.pi * progress))

    scheduler = torch.optim.lr_scheduler.LambdaLR(optimiser, schedule)

    history, best = [], {"macro_f1": -1.0, "epoch": -1, "state": None, "logits": None}
    start = time.time()

    for epoch in range(epochs):
        epoch_start = time.time()
        train_loss, _ = run_epoch(model, train_loader, criterion, optimiser, logit_bias, scaler)
        val_loss, val_logits = run_epoch(model, val_loader, criterion, None, None, None)
        scheduler.step()

        val_pred = val_logits.argmax(axis=1)
        macro = f1_score(y_val, val_pred, labels=SCOREABLE, average="macro", zero_division=0)
        history.append({
            "epoch": epoch + 1, "train loss": train_loss, "val loss": val_loss,
            "val accuracy": accuracy_score(y_val, val_pred), "val macro-F1": macro,
            "lr": optimiser.param_groups[0]["lr"], "seconds": time.time() - epoch_start,
        })

        if macro > best["macro_f1"]:
            best.update({
                "macro_f1": macro, "epoch": epoch + 1,
                "state": {k: v.detach().cpu().clone() for k, v in model.state_dict().items()},
                "logits": val_logits,
            })

        # Every epoch by default. Silence across several minutes is indistinguishable from a
        # hang, which is what made the original five-epoch reporting interval a poor default.
        if verbose_every and ((epoch + 1) % verbose_every == 0 or epoch == 0):
            print(f"  [{label}] epoch {epoch + 1:>3}/{epochs}  train {train_loss:.3f}  "
                  f"val {val_loss:.3f}  acc {history[-1]['val accuracy']:.3f}  "
                  f"macro-F1 {macro:.4f}  {history[-1]['seconds']:.0f}s")

        if epoch + 1 - best["epoch"] >= patience:
            print(f"  [{label}] early stop at epoch {epoch + 1}; best was epoch {best['epoch']}")
            break

    model.load_state_dict(best["state"])   # report and save the same weights
    print(f"  [{label}] best macro-F1 {best['macro_f1']:.4f} at epoch {best['epoch']} "
          f"({time.time() - start:.0f}s total, "
          f"{np.mean([h['seconds'] for h in history]):.0f}s per epoch)")
    return pd.DataFrame(history), best["logits"], best["macro_f1"]


def bank(name, model, logits):
    """Write a trained model to disk as soon as it exists.

    Section 9 is the last thing in the notebook, which means a run interrupted anywhere before
    it loses every model it had already trained. Checkpointing here costs a second and makes
    each model independently recoverable.
    """
    ARTEFACT_DIR.mkdir(parents=True, exist_ok=True)
    path = ARTEFACT_DIR / f"checkpoint_{name}.pt"
    torch.save({
        "name": name,
        "state_dict": {k: v.detach().cpu() for k, v in model.state_dict().items()},
        "val_logits": logits,
        "classes": CLASSES,
        "normalisation_mean": NORM_MEAN.tolist(),
        "normalisation_std": NORM_STD.tolist(),
    }, path)
    print(f"  banked -> {path}")


def plot_history(histories, title):
    """Training curves for one or more runs."""
    fig, axes = plt.subplots(1, 2, figsize=(12, 3.8))
    for (label, history), colour in zip(histories.items(), PALETTE):
        axes[0].plot(history["epoch"], history["train loss"], color=colour, label=f"{label} train")
        axes[0].plot(history["epoch"], history["val loss"], color=colour, ls="--",
                     label=f"{label} val")
        axes[1].plot(history["epoch"], history["val macro-F1"], color=colour, label=label)
    axes[0].set(title="Loss", xlabel="Epoch", ylabel="Cross-entropy")
    axes[1].set(title="Validation macro-F1", xlabel="Epoch", ylabel="Macro-F1")
    for ax in axes:
        ax.legend(fontsize=8)
    fig.suptitle(title, y=1.03)
    plt.tight_layout(); plt.show()


def count_parameters(model):
    return sum(p.numel() for p in model.parameters() if p.requires_grad)

## 5. Model 2: CNN from Scratch

The reference point, and deliberately plain: stacked 3x3 convolutions with batch normalisation,
max pooling, global average pooling, one linear head, plain cross-entropy. No residual
connections, no imbalance handling, no schedule tricks beyond the shared recipe. Keeping it plain
is what makes the third model's gain attributable to the two changes that model introduces rather
than to an accumulation of unrelated improvements.

Three design decisions follow from the input geometry.

- **Three pooling stages, not five.** At 80x60 the spatial map after three halvings is 10x7. A
  fourth would leave 5x3, at which point the convolutions are operating on almost no spatial
  structure. Depth is added by stacking convolutions within a stage instead.
- **Batch normalisation on every convolution.** It is what makes this network trainable at
  lr = 1e-3 within a 40-epoch budget rather than requiring a careful initialisation search.
- **Global average pooling instead of flatten-and-dense.** A flatten at 10x7x256 would put
  roughly 2.2M parameters in the classifier alone, most of the network, and would overfit the
  head classes. Average pooling keeps the parameter budget in the feature extractor.

In [ ]:
class PlainCNN(nn.Module):
    """VGG-style stack sized for 60x80 inputs. The reference architecture."""

    def __init__(self, n_classes=N_CLASSES, width=32, dropout=0.3):
        super().__init__()

        def block(in_channels, out_channels, pool=True):
            layers = [
                nn.Conv2d(in_channels, out_channels, 3, padding=1, bias=False),
                nn.BatchNorm2d(out_channels), nn.ReLU(inplace=True),
                nn.Conv2d(out_channels, out_channels, 3, padding=1, bias=False),
                nn.BatchNorm2d(out_channels), nn.ReLU(inplace=True),
            ]
            if pool:
                layers.append(nn.MaxPool2d(2))
            return nn.Sequential(*layers)

        self.features = nn.Sequential(
            block(3, width),                 # 80x60 -> 40x30
            block(width, width * 2),         # 40x30 -> 20x15
            block(width * 2, width * 4),     # 20x15 -> 10x7
            block(width * 4, width * 8, pool=False),
        )
        self.pool = nn.AdaptiveAvgPool2d(1)
        self.dropout = nn.Dropout(dropout)
        self.fc = nn.Linear(width * 8, n_classes)

    def forward(self, x):
        x = self.pool(self.features(x)).flatten(1)
        return self.fc(self.dropout(x))


set_seed(RANDOM_STATE)
cnn = PlainCNN()
print(f"PlainCNN parameters: {count_parameters(cnn):,}")
print("Output shape check:", tuple(cnn(torch.zeros(2, 3, IMAGE_TARGET_SIZE[1], IMAGE_TARGET_SIZE[0]))
                                   .shape))

In [ ]:
set_seed(RANDOM_STATE)
cnn = prepare_model(PlainCNN())
cnn_history, cnn_logits, _ = fit(cnn, train_loader, val_loader, label="CNN")

cnn_pred = cnn_logits.argmax(axis=1)
record(evaluate_predictions(y_val, cnn_pred, cnn_logits, "2. CNN from scratch"))
bank("cnn", cnn, cnn_logits)
plot_history({"CNN": cnn_history}, "Model 2: plain CNN")
report_memory("after CNN")

## 6. Model 3: Small ResNet with Decoupled Classifier Retraining

The advanced model makes exactly two changes to model 2, each targeting a different weakness, and
Section 6.3 measures them separately so the report can attribute the gain rather than assert it.

**Change 1, the architecture.** Residual connections, four stages, and a stem adapted to this
input. The standard ImageNet stem is a 7x7 stride-2 convolution followed by a stride-2 max pool,
which reduces a 224-pixel input by a factor of four before the first residual block. Applied to an
80-pixel image that leaves 20x15 immediately and discards most of the spatial detail the task
depends on. This implementation uses the CIFAR-style stem instead: a single 3x3 stride-1
convolution, with all downsampling performed inside the stages.

**Change 2, the objective.** Kang et al. (ICLR 2020) report that on long-tailed data the feature
extractor learns best from the natural, instance-balanced distribution, while the classifier is
what needs rebalancing. That decomposition suggests training in two stages: fit the whole network
normally, then freeze the backbone, reinitialise the final linear layer, and retrain only that
layer under class-balanced sampling. It is a few hundred parameters' worth of retraining and it
typically moves macro-F1 more than any architectural change.

Section 6.3 also evaluates logit-adjusted cross-entropy (Menon et al., ICLR 2021), a one-line
alternative that adds `tau * log(prior)` to the logits during training, as a competing treatment
for the same problem.

In [ ]:
class BasicBlock(nn.Module):
    """Standard two-convolution residual block with an optional projection shortcut."""

    def __init__(self, in_channels, out_channels, stride=1):
        super().__init__()
        self.conv1 = nn.Conv2d(in_channels, out_channels, 3, stride, 1, bias=False)
        self.bn1 = nn.BatchNorm2d(out_channels)
        self.conv2 = nn.Conv2d(out_channels, out_channels, 3, 1, 1, bias=False)
        self.bn2 = nn.BatchNorm2d(out_channels)
        self.shortcut = nn.Sequential()
        if stride != 1 or in_channels != out_channels:
            self.shortcut = nn.Sequential(
                nn.Conv2d(in_channels, out_channels, 1, stride, bias=False),
                nn.BatchNorm2d(out_channels),
            )

    def forward(self, x):
        out = F.relu(self.bn1(self.conv1(x)), inplace=True)
        out = self.bn2(self.conv2(out))
        return F.relu(out + self.shortcut(x), inplace=True)


class SmallResNet(nn.Module):
    """ResNet-18 topology with a stride-1 3x3 stem, sized for 60x80 inputs.

    The backbone and the classifier are separate attributes so that Section 6.2 can freeze
    one and replace the other without touching the forward pass.
    """

    def __init__(self, n_classes=N_CLASSES, width=64, blocks=(2, 2, 2, 2), dropout=0.2):
        super().__init__()
        self.stem = nn.Sequential(
            nn.Conv2d(3, width, 3, 1, 1, bias=False),
            nn.BatchNorm2d(width), nn.ReLU(inplace=True),
        )
        stages, in_channels = [], width
        for stage_index, n_blocks in enumerate(blocks):
            out_channels = width * (2 ** stage_index)
            for block_index in range(n_blocks):
                stride = 2 if (block_index == 0 and stage_index > 0) else 1
                stages.append(BasicBlock(in_channels, out_channels, stride))
                in_channels = out_channels
        self.stages = nn.Sequential(*stages)
        self.pool = nn.AdaptiveAvgPool2d(1)
        self.dropout = nn.Dropout(dropout)
        self.feature_dim = in_channels
        self.fc = nn.Linear(in_channels, n_classes)

    def embed(self, x):
        """Penultimate features. Used by the decoupled stage and by the saliency check."""
        return self.pool(self.stages(self.stem(x))).flatten(1)

    def forward(self, x):
        return self.fc(self.dropout(self.embed(x)))


set_seed(RANDOM_STATE)
resnet = SmallResNet()
print(f"SmallResNet parameters: {count_parameters(resnet):,}")
print("Output shape check:", tuple(resnet(torch.zeros(2, 3, IMAGE_TARGET_SIZE[1],
                                                      IMAGE_TARGET_SIZE[0])).shape))

### 6.1 Stage 1: Instance-Balanced Training

In [ ]:
set_seed(RANDOM_STATE)
resnet = prepare_model(SmallResNet())
resnet_history, resnet_logits, _ = fit(resnet, train_loader, val_loader, label="ResNet stage 1")

resnet_pred = resnet_logits.argmax(axis=1)
resnet_stage1 = record(evaluate_predictions(
    y_val, resnet_pred, resnet_logits, "3a. ResNet (plain CE)"
))
bank("resnet_stage1", resnet, resnet_logits)
plot_history({"CNN": cnn_history, "ResNet": resnet_history}, "Model 2 against model 3, stage 1")
report_memory("after ResNet stage 1")

### 6.2 Stage 2: Class-Balanced Classifier Retraining

The backbone is frozen and held in evaluation mode, so its batch-normalisation running statistics
do not drift under the resampled distribution; only the reinitialised final linear layer trains.
The sampler draws each class with probability proportional to `1 / support`, so a batch is
approximately class-balanced rather than image-balanced.

This is deliberately cheap. It touches roughly 0.4% of the network's parameters for a tenth of the
stage-1 epoch budget, which is the point: if it produces a material macro-F1 gain, that is direct
evidence for the claim that the classifier, not the representation, was the bottleneck.

In [ ]:
# Inverse-support weights, one per training row. Classes with a single example are drawn as
# often as Tshirts, which is the intended behaviour of a class-balanced sampler.
class_weights = 1.0 / np.maximum(np.bincount(y_train, minlength=N_CLASSES), 1)
sample_weights = class_weights[y_train]
balanced_loader, _ = make_loaders(weights=sample_weights)

# Freeze everything, then replace the head. eval() on the whole model keeps BatchNorm in
# inference mode; the new linear layer has no BatchNorm of its own, so nothing is lost.
decoupled = prepare_model(SmallResNet())
decoupled.load_state_dict(resnet.state_dict())
for parameter in decoupled.parameters():
    parameter.requires_grad = False
decoupled.fc = nn.Linear(decoupled.feature_dim, N_CLASSES).to(DEVICE)
decoupled.eval()

trainable = count_parameters(decoupled)
print(f"Trainable parameters in stage 2: {trainable:,} "
      f"({trainable / sum(p.numel() for p in decoupled.parameters()) * 100:.2f}% of the model)")


def run_stage_two(model, loader, epochs=STAGE2_EPOCHS, lr=STAGE2_LR):
    """Retrain the classifier only, under class-balanced sampling.

    The backbone runs under no_grad, so this pass costs a forward only. It is the cheapest
    thing in the notebook and usually the largest single macro-F1 gain.
    """
    criterion = nn.CrossEntropyLoss(label_smoothing=LABEL_SMOOTHING)
    optimiser = torch.optim.AdamW(model.fc.parameters(), lr=lr, weight_decay=WEIGHT_DECAY)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimiser, T_max=epochs)
    scaler = make_scaler()
    best = {"macro_f1": -1.0, "state": None, "logits": None}

    for epoch in range(epochs):
        start = time.time()
        model.eval()                      # backbone stays in inference mode throughout
        model.fc.train()
        for images, labels in loader:
            with torch.autocast(device_type=DEVICE.type, dtype=AMP_DTYPE, enabled=AMP_ENABLED):
                with torch.no_grad():
                    features = model.embed(images)
                loss = criterion(model.fc(features.float()), labels)
            optimiser.zero_grad(set_to_none=True)
            if scaler.is_enabled():
                scaler.scale(loss).backward()
                scaler.step(optimiser)
                scaler.update()
            else:
                loss.backward()
                optimiser.step()
        scheduler.step()

        _, logits = run_epoch(model, val_loader, criterion, None, None, None)
        macro = f1_score(y_val, logits.argmax(axis=1), labels=SCOREABLE,
                         average="macro", zero_division=0)
        print(f"  [stage 2] epoch {epoch + 1}/{epochs}  macro-F1 {macro:.4f}  "
              f"{time.time() - start:.0f}s")
        if macro > best["macro_f1"]:
            best.update({
                "macro_f1": macro,
                "state": {k: v.detach().cpu().clone() for k, v in model.state_dict().items()},
                "logits": logits,
            })

    model.load_state_dict(best["state"])
    return best["logits"], best["macro_f1"]


set_seed(RANDOM_STATE)
decoupled_logits, _ = run_stage_two(decoupled, balanced_loader)
decoupled_pred = decoupled_logits.argmax(axis=1)
record(evaluate_predictions(
    y_val, decoupled_pred, decoupled_logits, "3. ResNet + decoupled classifier"
))
bank("resnet_decoupled", decoupled, decoupled_logits)

### 6.3 Ablation: Which Change Paid?

Three treatments of the same imbalance, on the same architecture, the same split and the same
budget. This is an ablation table, not three more submissions: the point is to attribute the
macro-F1 gain, and to expose the trade-off it costs.

| Row | Treatment |
|---|---|
| Plain CE | Stage 1 alone. The reference. |
| Logit-adjusted CE | `tau * log(prior)` added to the logits during training only; evaluation uses the raw logits. One line, no extra training stage. |
| Decoupled classifier | Stage 1 followed by class-balanced retraining of the final layer. |

Expect the two rebalanced rows to raise macro-F1 and to *lower* top-1 accuracy: shifting decision
boundaries toward rare classes necessarily costs some head-class precision. That trade-off, and
the decision about which side of it to land on, is the substance of the ultimate judgement in
Section 8.

In [ ]:
# Logit-adjusted cross-entropy. The bias is added inside the loss during training and never
# at evaluation, so the deployed model is an ordinary softmax classifier.
log_prior = torch.log(
    torch.tensor(np.bincount(y_train, minlength=N_CLASSES) / len(y_train), dtype=torch.float32)
).to(DEVICE)

set_seed(RANDOM_STATE)
adjusted = prepare_model(SmallResNet())
adjusted_history, adjusted_logits, _ = fit(
    adjusted, train_loader, val_loader,
    logit_bias=LOGIT_ADJUST_TAU * log_prior, label="ResNet logit-adjusted",
)
adjusted_pred = adjusted_logits.argmax(axis=1)
record(evaluate_predictions(
    y_val, adjusted_pred, adjusted_logits, f"3b. ResNet (logit-adjusted, tau={LOGIT_ADJUST_TAU})"
))
bank("resnet_logit_adjusted", adjusted, adjusted_logits)

In [ ]:
ablation = pd.concat([r for r in RESULTS if r["Model"].iloc[0].startswith("3")], ignore_index=True)
display(ablation.style.format({c: "{:.4f}" for c in ablation.columns if c != "Model"}))

# The trade-off, drawn rather than described: accuracy against macro-F1 for every model so far.
comparison = pd.concat(RESULTS, ignore_index=True)
fig, ax = plt.subplots(figsize=(7.5, 5))
for (_, row), colour in zip(comparison.iterrows(), PALETTE * 3):
    ax.scatter(row["Top-1 accuracy"], row["Macro-F1"], s=90, color=colour, zorder=3)
    ax.annotate(row["Model"], (row["Top-1 accuracy"], row["Macro-F1"]),
                xytext=(6, 4), textcoords="offset points", fontsize=8)
ax.set(xlabel="Top-1 accuracy (image weighted)", ylabel="Macro-F1 (class weighted)",
       title="The head-tail trade-off across every model scored so far")
plt.tight_layout(); plt.show()

## 7. Seed Variance

With classes held by two or three groups, one validation example can move a per-class F1 from 0 to
1, and 124 such classes enter the macro average. A difference of a point or two between two models
may therefore be a difference in the random draw rather than in the models.

The two finalists are retrained under three seeds and reported as mean +/- standard deviation.
The seed governs weight initialisation, augmentation draws and sampler draws; **the split is held
fixed** so the comparison is on identical validation rows. That measures optimisation variance,
which is the variance a model choice is exposed to; split variance would require repeated
resplitting and is noted as a limitation in Section 10 rather than estimated here.

In [ ]:
def run_finalist(name, seed):
    """One full training run of a finalist under a given seed. Returns a scored row.

    The stream tensors are shared, so a seed run allocates a model and nothing else.
    """
    set_seed(seed)
    loader, _ = make_loaders()

    if name == "CNN":
        model = prepare_model(PlainCNN())
        _, logits, _ = fit(model, loader, val_loader, label=f"CNN s{seed}", verbose_every=10)
    else:
        model = prepare_model(SmallResNet())
        _, logits, _ = fit(model, loader, val_loader, label=f"ResNet s{seed}", verbose_every=10)
        balanced, _ = make_loaders(weights=sample_weights)
        for parameter in model.parameters():
            parameter.requires_grad = False
        model.fc = nn.Linear(model.feature_dim, N_CLASSES).to(DEVICE)
        logits, _ = run_stage_two(model, balanced)

    scored = evaluate_predictions(y_val, logits.argmax(axis=1), logits, name)
    scored["seed"] = seed

    del model
    gc.collect()
    if DEVICE.type == "cuda":
        torch.cuda.empty_cache()
    return scored


if RUN_SEED_STUDY:
    seed_rows = []
    for name in ["CNN", "ResNet + decoupled"]:
        for seed in SEEDS:
            print(f"--- {name}, seed {seed} ---")
            seed_rows.append(run_finalist(name, seed))
    seed_results = pd.concat(seed_rows, ignore_index=True)

    variance = seed_results.groupby("Model")[["Top-1 accuracy", "Macro-F1", "Balanced accuracy"]]
    summary = variance.agg(["mean", "std"]).round(4)
    display(summary)
    print("\nRead the standard deviation before claiming a winner: if the gap between the two "
          "means is inside one combined standard deviation, the models are not distinguishable "
          "on this evidence.")
else:
    seed_results = None
    print("RUN_SEED_STUDY is False. The comparison rests on a single seed and the report "
          "must say so.")

## 8. Ultimate Judgement

The metric table narrows the field; it does not settle it. This section applies five checks that a
performance number cannot answer, and the recommendation in Section 8.7 rests on all six pieces of
evidence together.

Assign the finalist here once, so every diagnostic below reads the same model.

In [ ]:
# The model taken forward. Change these three lines to re-run every diagnostic on a different
# candidate; nothing below hard-codes a model.
FINAL_NAME = "3. ResNet + decoupled classifier"
final_model = decoupled
final_logits = decoupled_logits
final_pred = final_logits.argmax(axis=1)

reference_name, reference_pred = "2. CNN from scratch", cnn_pred

summary_table = pd.concat(RESULTS, ignore_index=True)
display(summary_table.style.format({c: "{:.4f}" for c in summary_table.columns if c != "Model"})
        .background_gradient(subset=["Macro-F1"], cmap="Blues"))

### 8.1 Confusion Structure Against the EDA Hypotheses

Section 2.3 of notebook 00 ranked article types by the cosine similarity of their class mean
images and predicted which pairs a model would confuse, with Kurtas against Stoles the strongest
cross-family pair at 0.982. Those were hypotheses formed before any model existed. Checking them
against the realised confusion matrix closes the loop, and either outcome is informative:

- **Agreement** supports a visual explanation: the classes really are hard to separate at this
  resolution, and the error is a property of the data rather than of the model.
- **Disagreement** points elsewhere, to label noise, to imbalance, or to optimisation, and the
  remedy is different in each case.

Fill `EDA_HYPOTHESISED_PAIRS` from the cross-family table printed by Section 2.3 of notebook 00.

In [ ]:
# From Section 2.3 of the preprocessing notebook. Extend with the other leading cross-family
# pairs printed there.
EDA_HYPOTHESISED_PAIRS = [
    ("Kurtas", "Stoles"),
]

confusion = confusion_matrix(y_val, final_pred, labels=np.arange(N_CLASSES))

# Off-diagonal mass as a share of the true class's support: "when it is an X, how often is X
# called Y". Normalising by support stops head classes from dominating purely by volume.
with np.errstate(invalid="ignore", divide="ignore"):
    rates = confusion / confusion.sum(axis=1, keepdims=True)
rates = np.nan_to_num(rates)
np.fill_diagonal(rates, 0.0)

pairs = []
for true_index, predicted_index in zip(*np.where(rates > 0)):
    pairs.append({
        "True class": CLASSES[true_index],
        "Predicted as": CLASSES[predicted_index],
        "Errors": int(confusion[true_index, predicted_index]),
        "Share of true class %": rates[true_index, predicted_index] * 100,
        "Validation support": int(confusion[true_index].sum()),
    })
confused = (pd.DataFrame(pairs)
            .query("`Validation support` >= 10")
            .sort_values("Share of true class %", ascending=False)
            .head(15).reset_index(drop=True))
print("Most frequent confusions among classes with at least 10 validation images:")
display(confused.style.format({"Share of true class %": "{:.1f}%"}))

print("\nEDA hypotheses checked against the realised confusion matrix:")
for left, right in EDA_HYPOTHESISED_PAIRS:
    if left not in CLASS_TO_INDEX or right not in CLASS_TO_INDEX:
        print(f"  {left} / {right}: not both present in the training label space.")
        continue
    a, b = CLASS_TO_INDEX[left], CLASS_TO_INDEX[right]
    both_ways = confusion[a, b] + confusion[b, a]
    support = confusion[a].sum() + confusion[b].sum()
    print(f"  {left} <-> {right}: {both_ways} mutual errors out of {support} validation images "
          f"({both_ways / max(support, 1) * 100:.1f}%)")

In [ ]:
# The head of the label space, where the confusion matrix is dense enough to read. The full
# 124x124 matrix belongs in the appendix, not in a figure anyone is expected to interpret.
top_classes = train_support.sort_values(ascending=False).head(20).index
top_indices = [CLASS_TO_INDEX[c] for c in top_classes]
block = confusion[np.ix_(top_indices, top_indices)]
block_rates = block / np.maximum(block.sum(axis=1, keepdims=True), 1)

fig, ax = plt.subplots(figsize=(11, 9))
sns.heatmap(block_rates, xticklabels=top_classes, yticklabels=top_classes,
            cmap="Blues", vmin=0, vmax=1, square=True, ax=ax,
            cbar_kws={"label": "share of the true class"})
ax.set(title=f"{FINAL_NAME}: confusion among the 20 largest classes",
       xlabel="Predicted", ylabel="True")
plt.xticks(rotation=90, fontsize=8); plt.yticks(rotation=0, fontsize=8)
plt.tight_layout(); plt.show()

### 8.2 Per-Class F1 Against Support

Where does the model stop working, and at what support? The scatter answers a question the bucket
averages only summarise, and it is the figure that tells a reader how much labelled data a new
article type would need before the system could be trusted with it.

In [ ]:
final_f1 = per_class_f1(y_val, final_pred)
reference_f1 = per_class_f1(y_val, reference_pred)
support_scoreable = train_support.iloc[SCOREABLE]

fig, axes = plt.subplots(1, 2, figsize=(13, 4.6))
axes[0].scatter(support_scoreable, reference_f1, s=26, alpha=0.6,
                color=MUTED, label=reference_name)
axes[0].scatter(support_scoreable, final_f1, s=26, alpha=0.75,
                color=PALETTE[0], label=FINAL_NAME)
axes[0].set_xscale("log")
axes[0].set(title="Per-class F1 against training support", xlabel="Training images (log)",
            ylabel="Validation F1")
axes[0].legend(fontsize=8)

delta = (final_f1 - reference_f1).sort_values()
colours = [PALETTE[1] if value < 0 else PALETTE[2] for value in delta]
axes[1].bar(range(len(delta)), delta.values, color=colours)
axes[1].axhline(0, color=MUTED, lw=0.8)
axes[1].set(title=f"Per-class F1 change: {FINAL_NAME} minus {reference_name}",
            xlabel="Classes, ordered by change", ylabel="F1 difference")
plt.tight_layout(); plt.show()

print(f"Classes improved: {(delta > 0).sum()} | unchanged: {(delta == 0).sum()} | "
      f"degraded: {(delta < 0).sum()}")
print("\nLargest gains:"); print(delta.tail(8).round(3).to_string())
print("\nLargest losses:"); print(delta.head(8).round(3).to_string())
print(f"\nClasses scoring exactly zero F1: {(final_f1 == 0).sum()} of {len(final_f1)}")

### 8.3 Error Severity: Does the Model Stay in the Right Product Family?

Not every error costs the same. Predicting `Tshirts` for a `Tops` keeps the item inside Topwear,
where a shopper browsing that department still finds it; predicting `Watches` for a `Sarees` does
not. `subCategory` is available in the manifest as a diagnostic field, and Section 2.5 of notebook
00 permits exactly this use, so error severity can be measured without any metadata entering the
model.

A model with slightly lower accuracy but a higher share of within-family errors may well be the
better system to deploy, which is why this is part of the judgement rather than an aside.

In [ ]:
family_of_class = (
    frame.dropna(subset=[TARGET, "subCategory"])
    .groupby(TARGET)["subCategory"]
    .agg(lambda values: values.value_counts().index[0])
)
family_index = np.array([family_of_class.get(label, "unknown") for label in CLASSES])

severity_rows = []
for name, predictions in [(reference_name, reference_pred), (FINAL_NAME, final_pred)]:
    wrong = predictions != y_val
    same_family = family_index[predictions[wrong]] == family_index[y_val[wrong]]
    severity_rows.append({
        "Model": name,
        "Errors": int(wrong.sum()),
        "Error rate %": wrong.mean() * 100,
        "Within-family errors": int(same_family.sum()),
        "Within-family share of errors %": same_family.mean() * 100 if wrong.sum() else np.nan,
        "Cross-family error rate %": (wrong.sum() - same_family.sum()) / len(y_val) * 100,
    })
display(pd.DataFrame(severity_rows).style.format({
    "Error rate %": "{:.2f}%", "Within-family share of errors %": "{:.1f}%",
    "Cross-family error rate %": "{:.2f}%",
}))

### 8.4 Calibration

A catalogue tagger is only useful if it knows when it is unsure: the operating design is to
auto-apply confident predictions and route the rest to a human. That makes the reliability of the
confidence score a deployment property in its own right, separate from accuracy.

Expected Calibration Error is reported with a reliability diagram, and the coverage/accuracy curve
below turns it into the operational number: at what confidence threshold, and over what share of
the catalogue, is the system accurate enough to run unattended.

In [ ]:
def expected_calibration_error(probabilities, y_true, n_bins=15):
    """ECE with equal-width confidence bins, plus the per-bin data for the diagram."""
    confidence = probabilities.max(axis=1)
    predicted = probabilities.argmax(axis=1)
    correct = (predicted == y_true).astype(float)
    edges = np.linspace(0, 1, n_bins + 1)

    ece, rows = 0.0, []
    for low, high in zip(edges[:-1], edges[1:]):
        in_bin = (confidence > low) & (confidence <= high)
        if not in_bin.any():
            continue
        bin_accuracy, bin_confidence = correct[in_bin].mean(), confidence[in_bin].mean()
        ece += in_bin.mean() * abs(bin_accuracy - bin_confidence)
        rows.append({"confidence": bin_confidence, "accuracy": bin_accuracy,
                     "share": in_bin.mean()})
    return ece, pd.DataFrame(rows)


final_probabilities = torch.softmax(torch.from_numpy(final_logits), dim=1).numpy()
ece, reliability = expected_calibration_error(final_probabilities, y_val)
print(f"Expected Calibration Error: {ece:.4f}")

confidence = final_probabilities.max(axis=1)
correct = (final_pred == y_val)
thresholds = np.linspace(0.05, 0.99, 60)
coverage = [(confidence >= t).mean() for t in thresholds]
selective_accuracy = [
    correct[confidence >= t].mean() if (confidence >= t).any() else np.nan for t in thresholds
]

fig, axes = plt.subplots(1, 2, figsize=(12, 4.4))
axes[0].plot([0, 1], [0, 1], ls="--", color=MUTED, lw=1, label="perfect calibration")
axes[0].plot(reliability["confidence"], reliability["accuracy"], "o-", color=PALETTE[0],
             label="observed")
axes[0].set(title=f"Reliability diagram (ECE = {ece:.3f})", xlabel="Mean confidence",
            ylabel="Accuracy", xlim=(0, 1), ylim=(0, 1))
axes[0].legend(fontsize=8)

axes[1].plot(coverage, selective_accuracy, color=PALETTE[1], lw=2)
axes[1].set(title="Accuracy against coverage as the confidence threshold rises",
            xlabel="Share of the catalogue auto-tagged", ylabel="Accuracy on that share")
plt.tight_layout(); plt.show()

for target_accuracy in [0.95, 0.98]:
    reachable = [c for c, a in zip(coverage, selective_accuracy)
                 if not np.isnan(a) and a >= target_accuracy]
    if reachable:
        print(f"{target_accuracy:.0%} accuracy is reachable on {max(reachable):.1%} "
              "of the catalogue without human review.")
    else:
        print(f"{target_accuracy:.0%} accuracy is not reachable at any confidence threshold.")

### 8.5 Deployment Cost

Parameter count, checkpoint size and inference latency. The HD/DI extension asks for the model to
sit behind an interface rather than a terminal, which makes single-image latency a design
constraint and not a footnote.

In [ ]:
def measure_latency(model, batch_size, repeats=20):
    model.eval()
    dummy = torch.randn(batch_size, 3, IMAGE_TARGET_SIZE[1], IMAGE_TARGET_SIZE[0], device=DEVICE)
    with torch.no_grad():
        for _ in range(3):                       # warm up kernels and caches
            model(dummy)
        if DEVICE.type == "cuda":
            torch.cuda.synchronize()
        start = time.time()
        for _ in range(repeats):
            model(dummy)
        if DEVICE.type == "cuda":
            torch.cuda.synchronize()
    return (time.time() - start) / repeats / batch_size * 1000


cost_rows = []
for name, model in [(reference_name, cnn), (FINAL_NAME, final_model)]:
    parameters = count_parameters(model)
    cost_rows.append({
        "Model": name,
        "Parameters": parameters,
        "Checkpoint size (MB)": parameters * 4 / 1e6,
        "ms per image (batch 1)": measure_latency(model, 1),
        "ms per image (batch 128)": measure_latency(model, 128),
    })
display(pd.DataFrame(cost_rows).style.format({
    "Parameters": "{:,.0f}", "Checkpoint size (MB)": "{:.1f}",
    "ms per image (batch 1)": "{:.2f}", "ms per image (batch 128)": "{:.3f}",
}))
print(f"Measured on {DEVICE}. A CPU deployment target must be timed on CPU before the "
      "latency figure is quoted in the report.")

### 8.6 What the Model Looks At

Gradient saliency on a handful of correct and incorrect predictions. The specific question is
whether the model uses the garment or the studio background: Section 1.5 of notebook 00 noted that
the white background is shared across classes, so any model relying on it is exploiting framing
rather than product appearance and would fail on differently photographed stock.

This is a qualitative check, not evidence of a causal mechanism. It is included because it can
falsify a specific failure mode cheaply, and a few of these panels make a strong appendix figure.

In [ ]:
if RUN_SALIENCY:
    def saliency_map(model, image_normalised):
        """Absolute gradient of the top logit with respect to the input, max over channels.

        Run in full precision: autocast would quantise the gradient the figure is showing.
        """
        model.eval()
        image = image_normalised.unsqueeze(0).clone().requires_grad_(True)
        logits = model(image)
        logits[0, logits.argmax()].backward()
        return image.grad.abs().max(dim=1)[0].squeeze().detach().cpu().numpy()

    correct_positions = np.flatnonzero(final_pred == y_val)
    wrong_positions = np.flatnonzero(final_pred != y_val)
    picker = np.random.RandomState(RANDOM_STATE)
    chosen = np.concatenate([
        picker.choice(correct_positions, 4, replace=False),
        picker.choice(wrong_positions, 4, replace=False),
    ])

    index = torch.as_tensor(chosen, device=val_loader.images.device)
    raw = val_loader.images[index].to(DEVICE).permute(0, 3, 1, 2).float().div(255.0)
    normalised = (raw - NORM_MEAN_T) / NORM_STD_T

    fig, axes = plt.subplots(2, 8, figsize=(16, 4.8))
    for column, position in enumerate(chosen):
        axes[0, column].imshow(to_displayable(raw[column]))
        axes[0, column].set_title(
            f"true {CLASSES[y_val[position]]}\npred {CLASSES[final_pred[position]]}",
            fontsize=6.5,
            color=("#1baf7a" if final_pred[position] == y_val[position] else "#eb6834"),
        )
        axes[1, column].imshow(saliency_map(final_model, normalised[column]), cmap="inferno")
        for row in range(2):
            axes[row, column].axis("off")
    fig.suptitle("Input (top) and gradient saliency (bottom): four correct, then four errors",
                 y=1.04)
    plt.tight_layout(); plt.show()
else:
    print("RUN_SALIENCY is False.")

### 8.7 The Judgement

Write the recommendation here, against the evidence assembled above rather than against the
headline metric alone. The template below names the six inputs; replace each bracket with the
figure this run actually produced.

> **Recommendation: [model].**
>
> 1. **Aggregate performance.** It reaches [macro-F1] macro-F1 against [x] for the plain CNN and
>    [y] for the HOG baseline, at [z] top-1 accuracy. The majority-class baseline reaches
>    [accuracy] accuracy at essentially zero macro-F1, which is the reason the macro column is
>    the one being read.
> 2. **Where the gain sits.** The bucket breakdown attributes the improvement to the
>    [tail / body] classes, with the head effectively unchanged. The classifier retraining stage
>    accounts for [n] points of it, which supports the Kang et al. claim that the classifier and
>    not the representation was the bottleneck on this distribution.
> 3. **What it costs.** Top-1 accuracy falls by [n] points relative to plain cross-entropy. The
>    trade is [defensible / not defensible] for a catalogue system, where an untagged rare
>    article type is a product that cannot be found at all, while a misfiled common item is
>    still reachable through its neighbours.
> 4. **Error severity.** [n]% of its errors stay inside the correct `subCategory`, against [n]%
>    for the reference model, so its mistakes are systematically cheaper as well as fewer.
> 5. **Reliability.** ECE of [n] means the confidence score can carry an operating threshold:
>    [n]% of the catalogue can be auto-tagged at [n]% accuracy, with the remainder routed for
>    review. This is the property that makes the system deployable rather than merely accurate.
> 6. **Stability.** Across three seeds the macro-F1 spread is [n], so the margin over the
>    reference model [is / is not] larger than the run-to-run variation.
>
> **Where it should not be trusted.** [n] classes score zero F1, all with fewer than [n] training
> images, and [n] further classes have no validation example at all and are therefore unmeasured.
> `Suits` cannot be predicted, as its only record has no image (Section 1.4 of notebook 00). New
> article types should be routed to human review until they reach roughly [n] examples.

## 9. Persisting the Model and Producing Predictions

Four artefacts, and all four are required for the prediction to be reproducible: the weights, the
class-index mapping, the normalisation constants, and the configuration that produced them.
Losing the mapping alone would silently permute every prediction, since nothing in the checkpoint
records which output index means which article type.

In [ ]:
ARTEFACT_DIR.mkdir(parents=True, exist_ok=True)
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

checkpoint = {
    "model_name": FINAL_NAME,
    "architecture": type(final_model).__name__,
    "state_dict": final_model.state_dict(),
    "classes": CLASSES,
    "normalisation_mean": NORM_MEAN.tolist(),
    "normalisation_std": NORM_STD.tolist(),
    "image_target_size": list(IMAGE_TARGET_SIZE),
    "config": {
        "random_state": RANDOM_STATE, "validation_share": VALIDATION_SHARE,
        "batch_size": BATCH_SIZE, "epochs": EPOCHS, "learning_rate": LEARNING_RATE,
        "weight_decay": WEIGHT_DECAY, "label_smoothing": LABEL_SMOOTHING,
        "stage2_epochs": STAGE2_EPOCHS, "stage2_lr": STAGE2_LR,
    },
}
torch.save(checkpoint, ARTEFACT_DIR / "task1_model.pt")
(ARTEFACT_DIR / "task1_classes.json").write_text(json.dumps(CLASSES, indent=2))
(ARTEFACT_DIR / "task1_config.json").write_text(json.dumps(checkpoint["config"], indent=2))
summary_table.to_csv(ARTEFACT_DIR / "task1_results.csv", index=False)

print("Saved to", ARTEFACT_DIR.resolve())
for path in sorted(ARTEFACT_DIR.iterdir()):
    print(f"  {path.name}  ({path.stat().st_size / 1e6:.1f} MB)")

In [ ]:
# Every id in the template receives a prediction. The test images pass through the identical
# deterministic transform and the identical training-fitted normalisation constants; no test
# statistic is computed and nothing here is fitted.
template = pd.read_csv(PREDICTION_TEMPLATE)
print(f"Template rows: {len(template):,} | columns: {list(template.columns)}")

test_paths = [str(Path(TEST_IMAGE_DIR) / f"{image_id}.jpg") for image_id in template["id"]]
missing = [p for p in test_paths if not Path(p).exists()]
assert not missing, f"{len(missing)} test images are missing, e.g. {missing[:3]}"

width, height = IMAGE_TARGET_SIZE
X_test_images = np.empty((len(test_paths), height, width, 3), dtype=np.uint8)
for position, path in enumerate(test_paths):
    X_test_images[position] = load_image_array(path, target_size=IMAGE_TARGET_SIZE, scale=False)
print(f"Decoded {len(X_test_images):,} test images ({X_test_images.nbytes / 1e6:.0f} MB)")

test_loader = BatchStream(X_test_images, np.zeros(len(X_test_images), dtype=np.int64),
                          batch_size=512, augment=False, shuffle=False)

final_model.eval()
test_logits = []
with torch.no_grad():
    for images, _ in test_loader:
        with torch.autocast(device_type=DEVICE.type, dtype=AMP_DTYPE, enabled=AMP_ENABLED):
            test_logits.append(final_model(images).float().cpu())
test_logits = torch.cat(test_logits).numpy()

predictions = template.copy()
predictions[TARGET] = [CLASSES[index] for index in test_logits.argmax(axis=1)]

assert len(predictions) == len(template)
assert predictions[TARGET].notna().all()
assert set(predictions["id"]) == set(template["id"])

prediction_path = OUTPUT_DIR / "task1_predictions.csv"
predictions.to_csv(prediction_path, index=False)
print("Written:", prediction_path.resolve())
display(predictions.head())
print("\nPredicted class distribution (top 10):")
print(predictions[TARGET].value_counts().head(10).to_string())
print(f"\nDistinct classes predicted: {predictions[TARGET].nunique()} of {N_CLASSES} available")

#### A Note on the Predicted Distribution

Compare the predicted class distribution against the training distribution printed above. A model
that predicts only the head classes on the test set has not generalised its tail behaviour,
whatever its validation macro-F1 says, and the discrepancy is worth a sentence in the report.

Two qualifications carry over from notebook 00. Eleven test images are byte identical to training
images (Section 3.6), so any test-set claim is very slightly a memorisation score. And `Suits`
cannot appear in this output at all, because its single record has no image file.

The other three target columns are left as the template supplied them. Tasks 2 and 3 fill their
own columns, and the four are merged into a single submission file before upload.

### 9.1 Getting the Results off the Session Disk

Everything written so far lives under `/content` and disappears when the runtime disconnects. This
cell packages the checkpoint, the class mapping, the results table and the prediction file and
downloads them through the browser.

The prediction file is the submission artefact and the checkpoint is what lets an evaluator
reproduce it without retraining, so both must leave the session. Run this cell before closing the
tab. If the browser blocks the download, allow pop-ups for `colab.research.google.com` and re-run;
the files also remain in the sidebar file browser, where they can be downloaded one at a time from
the three-dot menu.

In [ ]:
from google.colab import files

# One archive for the model artefacts, plus the prediction CSV on its own, because the CSV is
# uploaded to a separate Canvas page and should not have to be unzipped first.
bundle_root = WORK_ROOT / "task1_bundle"
if bundle_root.exists():
    shutil.rmtree(bundle_root)
bundle_root.mkdir(parents=True)

for source in list(ARTEFACT_DIR.iterdir()) + [prediction_path]:
    shutil.copy(source, bundle_root / source.name)
    print(f"  bundled {source.name}  ({source.stat().st_size / 1e6:.1f} MB)")

bundle_path = shutil.make_archive(str(WORK_ROOT / "task1_artefacts"), "zip", str(bundle_root))
print("\nBundle:", bundle_path, f"({Path(bundle_path).stat().st_size / 1e6:.1f} MB)")

files.download(bundle_path)
files.download(str(prediction_path))
print("Downloads triggered. If nothing arrives, allow pop-ups for this site and re-run the cell.")

## 10. Decision Log, Limitations, and What to Tune Next

### 10.1 Decision Log

| Decision | Evidence | Section |
|---|---|---|
| Macro-F1 over the classes present in validation as the primary metric | 124 classes spanning 6,780:1; the majority baseline reaches high accuracy at near-zero macro-F1 | 2.5, 2.6 |
| Report macro-F1 by support bucket rather than one aggregate | Two models with equal macro-F1 can have opposite head/tail profiles | 2.1, 2.5 |
| Baseline is HOG + linear SVM, not only the majority class | Section 2.3 of notebook 00 found article types separate largely by silhouette, which HOG encodes directly | 3 |
| Model 2 kept deliberately plain | Makes model 3's gain attributable to its two named changes rather than to accumulated tricks | 5 |
| CIFAR-style stride-1 stem instead of the ImageNet stem | The 7x7 stride-2 stem plus max pool reduces an 80-pixel input to 20 pixels before the first block | 6 |
| Two-stage decoupled training | Kang et al. (2020): representations learn best instance-balanced, classifiers need rebalancing | 6.2 |
| Imbalance treatments run as ablation rows, not as separate submissions | The question is attribution of the gain, not a fourth model | 6.3 |
| Early stopping on validation macro-F1, not validation loss | Cross-entropy is dominated by head classes, so the minimum-loss epoch is not the best tail epoch | 4 |
| No random cropping in the augmentation policy | Section 3.1 of notebook 00: the portrait frame carries class cues at the top and bottom of a product | 2.4 |
| Three-seed reruns for the finalists | Classes with two or three validation examples make per-class F1, and hence macro-F1, high variance | 7 |

### 10.2 Limitations

- **Split variance is not estimated.** The three-seed study holds the split fixed and varies only
  optimisation, so it measures run-to-run variation and not the variation from a different
  validation draw. Repeated group-aware resplitting would estimate it and was not run for cost.
- **Classes absent from validation are unmeasured, not verified as working.** They are excluded
  from every macro average, so the headline slightly overstates coverage of the full label space.
- **Exact duplicate detection only.** Notebook 00 records that alternate views, colourways and
  reshoots of the same product are not detected by SHA-256 and can still cross the split, which
  would inflate validation performance by an unknown amount.
- **Resolution is a ceiling, not a choice.** Every result is at the catalogue's native 60x80. A
  larger tensor interpolates rather than adds detail, as Section 1.5 of notebook 00 argues.
- **Mixed precision is not free of numerical difference.** Results under bf16 or fp16 will not be bit-identical to an fp32 run, though the difference is far below the seed variance measured in Section 7. Set `USE_AMP = False` to reproduce an fp32 run exactly.\n- **Saliency is qualitative.** It can falsify a suspected background shortcut; it cannot establish
  what the model has actually learned.

### 10.3 Tuning Order

Tune in decreasing order of sensitivity, keeping the split, the metrics and the seed fixed. Log
every run and put the search table in the appendix; a reader should be able to see the space that
was searched, not only the configuration that won.

| Stage | What to sweep | Range | Why first |
|---|---|---|---|
| 1 | `LEARNING_RATE` | 3e-4, 1e-3, 3e-3 | Dominates every other hyperparameter here. Nothing else is worth tuning at a bad step size. |
| 2 | Capacity: `width` 32 vs 64, `blocks` (2,2,2,2) vs (1,1,2,2) | 2 configurations | Determines whether the model is capacity-limited or data-limited, which changes what stage 3 should do. |
| 3 | Regularisation: `dropout`, `WEIGHT_DECAY`, augmentation strength | dropout 0.2/0.3/0.5, decay 1e-4/1e-2 | Only meaningful once capacity is fixed. |
| 4 | `LOGIT_ADJUST_TAU`, `STAGE2_LR`, `STAGE2_EPOCHS` | tau 0.5/1.0/1.5 | The long-tail treatment, tuned last because it trades head accuracy for tail recall and that trade should be made against a settled model. |

Use random rather than grid search over stages 1-3, run the early stages on a stratified 30%
subsample of the training split to cut iteration time, and confirm the winning configuration on
the full split before it enters the comparison table.

### 10.4 Further Work

- **Test-time augmentation.** Average the logits over the image and its horizontal flip. Costs one
  extra forward pass and typically adds a fraction of a point; it needs no retraining.
- **Ensemble the finalists.** The seed-study checkpoints already exist, so averaging their softmax
  outputs is free and usually beats the best single member.
- **A pretrained comparison.** The brief permits pretrained models for comparison provided the
  submitted model is trained from scratch. A fine-tuned ResNet-18 quantifies how much of the gap
  to published results is attributable to representation transfer rather than to architecture.
- **Metric-learning objective.** Training the backbone under ArcFace instead of softmax
  cross-entropy would produce embeddings usable directly by the Task 4 retrieval system, giving
  one backbone across two deliverables.

### 10.5 References

- Kang, B. et al. (2020). Decoupling Representation and Classifier for Long-Tailed Recognition. *ICLR*.
- Menon, A. K. et al. (2021). Long-Tail Learning via Logit Adjustment. *ICLR*.
- Cui, Y. et al. (2019). Class-Balanced Loss Based on Effective Number of Samples. *CVPR*.
- He, K. et al. (2016). Deep Residual Learning for Image Recognition. *CVPR*.
- Dalal, N. and Triggs, B. (2005). Histograms of Oriented Gradients for Human Detection. *CVPR*.
- Szegedy, C. et al. (2016). Rethinking the Inception Architecture for Computer Vision. *CVPR*. (label smoothing)